# CogAttention — Thread Tracking (Capacity)

**Track:** Attention — Attention Capacity
**Benchmark:** CogAttention v1.0
**Task:** capacity

---

## Methodology

Tests attention capacity through multi-object tracking (Thread Tracking). N people each hold a unique item and swap in pairwise trades; the model must report who holds what after all swaps. Based on Pylyshyn's MOT paradigm (Pylyshyn & Storm, 1988).

### Cognitive Science Grounding

- **Multiple Object Tracking** (Pylyshyn & Storm, 1988): humans can track ~4 independent objects; we test whether LLMs hit similar capacity walls
- Models must maintain distinct identity-item bindings through pairwise swaps

### Difficulty Scaling

Easy: 2 people, 2 swaps | Medium: 3 people, 5 swaps | Hard: 4 people, 8 swaps | Expert: 5 people, 12 swaps | Frontier: 8 people, 25 swaps

### Scoring

One assertion per person — checks whether the model correctly reports each person's final item after all swaps.

All instances are procedurally generated from a seed with programmatic ground truth.
No static datasets. 7 layers of contamination resistance including canary strings,
zero lexical overlap (Semantic NIAH), and seed-based regeneration.

---

`<!-- COGATTENTION-BENCH-CANARY-0014FF929D70 -->`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Imports + Inline Helpers
# CogAttention — Attention Capacity
# ══════════════════════════════════════════════════════════════════════

import kaggle_benchmarks as kbench

import json
import re

def extract_answer_block(response):
    for pat in [r"ANSWER:\s*(.*)", r"Answer:\s*(.*)", r"answer:\s*(.*)"]:
        match = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return response.strip()

def extract_numbered_answers(response):
    answer_block = extract_answer_block(response)
    results = {}
    matches = re.findall(
        r"(\d+)\s*[.):\-]\s*(.+?)(?=\n\d+\s*[.):\-]|\Z)",
        answer_block, re.DOTALL,
    )
    for num, val in matches:
        results[num] = val.strip().rstrip(".")
    return results

def extract_list_items(response):
    answer_block = extract_answer_block(response)
    bullets = re.findall(r"[-\u2022]\s*(.+?)(?:\n|$)", answer_block)
    if bullets:
        return [b.strip().rstrip(".") for b in bullets]
    numeric_items = re.findall(
        r'[\$]?\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s*(?:\xb0[CF]|mg/L|%|\$))?',
        answer_block,
    )
    if numeric_items and len(numeric_items) >= 2:
        return [x.strip() for x in numeric_items]
    if "," in answer_block:
        items = [x.strip().rstrip(".") for x in answer_block.split(",")]
        return [x for x in items if x]
    lines = [l.strip().rstrip(".") for l in answer_block.split("\n") if l.strip()]
    return lines if lines else ([answer_block] if answer_block else [])

def extract_person_item_pairs(response):
    answer_block = extract_answer_block(response)
    results = {}
    for pat in [
        r"[-\u2022]?\s*(\w+)\s*:\s*(.+?)(?:\n|$)",
        r"[-\u2022]?\s*(\w+)\s+holds?\s+(?:a\s+)?(.+?)(?:\n|$)",
    ]:
        matches = re.findall(pat, answer_block, re.IGNORECASE)
        if matches:
            for name, item in matches:
                results[name.strip()] = item.strip().rstrip(".")
            break
    return results

def fuzzy_value_match(predicted, gold):
    pred_clean = re.sub(r"\s+", " ", predicted.strip().lower())
    gold_clean = re.sub(r"\s+", " ", gold.strip().lower())
    if pred_clean == gold_clean:
        return True
    if gold_clean in pred_clean:
        return True
    try:
        pred_num = float(re.sub(r"[,$%\xb0]", "", predicted))
        gold_num = float(re.sub(r"[,$%\xb0]", "", gold))
        return pred_num == gold_num
    except (ValueError, TypeError):
        pass
    return False

def _escape_for_regex(s):
    return re.escape(s).replace(r"\ ", r"\s+")


def run_assertions_capacity(response, gold, kbench):
    for person in gold["people"]:
        gold_item = gold["answers"][person]
        pattern = rf"(?i){re.escape(person)}\s*[:.\\-]\s*.*{_escape_for_regex(gold_item)}"
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"{person} should hold '{gold_item}'"
        )


print("CogAttention helpers loaded")
print(f"Task types: ['capacity']")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Task Definitions + Embedded Dataset
# ══════════════════════════════════════════════════════════════════════


@kbench.task(name="cogattention_capacity")
def cogattention_capacity(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention capacity task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_capacity(response, gold, kbench)


# ── Embedded dataset ──────────────────────────────────────────────────
DATASET = json.loads(r'''
[
 {
  "task_id": "capacity_easy_000",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Paloma holds a ivory flask\n- Idris holds a green feather\n\nSwaps:\n1. Paloma and Idris swap items.\n2. Paloma and Idris swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Paloma\": \"ivory flask\", \"Idris\": \"green feather\"}, \"people\": [\"Paloma\", \"Idris\"]}"
 },
 {
  "task_id": "capacity_easy_001",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Dariush holds a cobalt mask\n- Tala holds a coral flask\n\nSwaps:\n1. Dariush and Tala swap items.\n2. Dariush and Tala swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Dariush\": \"cobalt mask\", \"Tala\": \"coral flask\"}, \"people\": [\"Dariush\", \"Tala\"]}"
 },
 {
  "task_id": "capacity_easy_002",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Xander holds a russet coin\n- Ugo holds a copper pendant\n\nSwaps:\n1. Xander and Ugo swap items.\n2. Xander and Ugo swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Xander\": \"russet coin\", \"Ugo\": \"copper pendant\"}, \"people\": [\"Xander\", \"Ugo\"]}"
 },
 {
  "task_id": "capacity_easy_003",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Dariush holds a indigo locket\n- Nalini holds a slate shell\n\nSwaps:\n1. Dariush and Nalini swap items.\n2. Dariush and Nalini swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Dariush\": \"indigo locket\", \"Nalini\": \"slate shell\"}, \"people\": [\"Dariush\", \"Nalini\"]}"
 },
 {
  "task_id": "capacity_easy_004",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Tariq holds a golden coin\n- Wren holds a blue candle\n\nSwaps:\n1. Tariq and Wren swap items.\n2. Tariq and Wren swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Tariq\": \"golden coin\", \"Wren\": \"blue candle\"}, \"people\": [\"Tariq\", \"Wren\"]}"
 },
 {
  "task_id": "capacity_easy_005",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Greta holds a copper chalice\n- Tala holds a golden shell\n\nSwaps:\n1. Greta and Tala swap items.\n2. Greta and Tala swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Greta\": \"copper chalice\", \"Tala\": \"golden shell\"}, \"people\": [\"Greta\", \"Tala\"]}"
 },
 {
  "task_id": "capacity_easy_006",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Lumi holds a russet shell\n- Joaquin holds a slate stone\n\nSwaps:\n1. Lumi and Joaquin swap items.\n2. Lumi and Joaquin swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Lumi\": \"russet shell\", \"Joaquin\": \"slate stone\"}, \"people\": [\"Lumi\", \"Joaquin\"]}"
 },
 {
  "task_id": "capacity_easy_007",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Colette holds a onyx mirror\n- Tala holds a teal lantern\n\nSwaps:\n1. Colette and Tala swap items.\n2. Colette and Tala swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Colette\": \"onyx mirror\", \"Tala\": \"teal lantern\"}, \"people\": [\"Colette\", \"Tala\"]}"
 },
 {
  "task_id": "capacity_medium_008",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Bram holds a jade bell\n- Zain holds a golden flask\n- Zora holds a indigo compass\n\nSwaps:\n1. Zain and Zora swap items.\n2. Bram and Zora swap items.\n3. Zain and Zora swap items.\n4. Bram and Zain swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Bram\": \"jade bell\", \"Zain\": \"golden flask\", \"Zora\": \"indigo compass\"}, \"people\": [\"Bram\", \"Zain\", \"Zora\"]}"
 },
 {
  "task_id": "capacity_medium_009",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Haruto holds a green compass\n- Sigrid holds a slate coin\n- Uma holds a crimson flask\n\nSwaps:\n1. Haruto and Uma swap items.\n2. Sigrid and Uma swap items.\n3. Haruto and Sigrid swap items.\n4. Sigrid and Uma swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Haruto\": \"green compass\", \"Sigrid\": \"slate coin\", \"Uma\": \"crimson flask\"}, \"people\": [\"Haruto\", \"Sigrid\", \"Uma\"]}"
 },
 {
  "task_id": "capacity_medium_010",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Bram holds a cobalt key\n- Soren holds a onyx candle\n- Elio holds a silver dagger\n\nSwaps:\n1. Soren and Elio swap items.\n2. Bram and Soren swap items.\n3. Soren and Elio swap items.\n4. Bram and Elio swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Bram\": \"cobalt key\", \"Soren\": \"onyx candle\", \"Elio\": \"silver dagger\"}, \"people\": [\"Bram\", \"Soren\", \"Elio\"]}"
 },
 {
  "task_id": "capacity_medium_011",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Amara holds a cobalt feather\n- Celine holds a ochre shell\n- Nico holds a slate locket\n\nSwaps:\n1. Amara and Celine swap items.\n2. Amara and Nico swap items.\n3. Celine and Nico swap items.\n4. Amara and Nico swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Amara\": \"cobalt feather\", \"Celine\": \"ochre shell\", \"Nico\": \"slate locket\"}, \"people\": [\"Amara\", \"Celine\", \"Nico\"]}"
 },
 {
  "task_id": "capacity_medium_012",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Paloma holds a slate bell\n- Greta holds a blue key\n- Yuki holds a jade dagger\n\nSwaps:\n1. Paloma and Greta swap items.\n2. Paloma and Yuki swap items.\n3. Paloma and Greta swap items.\n4. Paloma and Yuki swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Paloma\": \"blue key\", \"Greta\": \"jade dagger\", \"Yuki\": \"slate bell\"}, \"people\": [\"Paloma\", \"Greta\", \"Yuki\"]}"
 },
 {
  "task_id": "capacity_medium_013",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Joelle holds a red key\n- Freya holds a ochre pendant\n- Lumi holds a copper stone\n\nSwaps:\n1. Joelle and Lumi swap items.\n2. Joelle and Freya swap items.\n3. Freya and Lumi swap items.\n4. Joelle and Freya swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Joelle\": \"red key\", \"Freya\": \"ochre pendant\", \"Lumi\": \"copper stone\"}, \"people\": [\"Joelle\", \"Freya\", \"Lumi\"]}"
 },
 {
  "task_id": "capacity_medium_014",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Qadir holds a green mirror\n- Maren holds a slate mask\n- Greta holds a cobalt shell\n\nSwaps:\n1. Maren and Greta swap items.\n2. Qadir and Greta swap items.\n3. Qadir and Maren swap items.\n4. Maren and Greta swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Qadir\": \"cobalt shell\", \"Maren\": \"green mirror\", \"Greta\": \"slate mask\"}, \"people\": [\"Qadir\", \"Maren\", \"Greta\"]}"
 },
 {
  "task_id": "capacity_medium_015",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Ravi holds a coral chalice\n- Lumi holds a copper ring\n- Yara holds a jade feather\n\nSwaps:\n1. Lumi and Yara swap items.\n2. Ravi and Lumi swap items.\n3. Ravi and Yara swap items.\n4. Lumi and Yara swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Ravi\": \"copper ring\", \"Lumi\": \"jade feather\", \"Yara\": \"coral chalice\"}, \"people\": [\"Ravi\", \"Lumi\", \"Yara\"]}"
 },
 {
  "task_id": "capacity_hard_016",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Dariush holds a cobalt locket\n- Sigrid holds a onyx ring\n- Freya holds a russet flask\n- Olena holds a teal key\n\nSwaps:\n1. Dariush and Olena swap items.\n2. Dariush and Sigrid swap items.\n3. Sigrid and Olena swap items.\n4. Dariush and Olena swap items.\n5. Freya and Olena swap items.\n6. Sigrid and Freya swap items.\n7. Sigrid and Olena swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Dariush\": \"teal key\", \"Sigrid\": \"russet flask\", \"Freya\": \"cobalt locket\", \"Olena\": \"onyx ring\"}, \"people\": [\"Dariush\", \"Sigrid\", \"Freya\", \"Olena\"]}"
 },
 {
  "task_id": "capacity_hard_017",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Viktor holds a pearl coin\n- Ines holds a teal candle\n- Amara holds a jade ring\n- Orla holds a slate chalice\n\nSwaps:\n1. Amara and Orla swap items.\n2. Ines and Orla swap items.\n3. Viktor and Amara swap items.\n4. Viktor and Ines swap items.\n5. Viktor and Orla swap items.\n6. Ines and Amara swap items.\n7. Ines and Orla swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Viktor\": \"teal candle\", \"Ines\": \"jade ring\", \"Amara\": \"slate chalice\", \"Orla\": \"pearl coin\"}, \"people\": [\"Viktor\", \"Ines\", \"Amara\", \"Orla\"]}"
 },
 {
  "task_id": "capacity_hard_018",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Dmitri holds a silver ring\n- Viktor holds a ochre mirror\n- Yara holds a copper dice\n- Tariq holds a indigo coin\n\nSwaps:\n1. Viktor and Tariq swap items.\n2. Viktor and Yara swap items.\n3. Yara and Tariq swap items.\n4. Dmitri and Viktor swap items.\n5. Dmitri and Tariq swap items.\n6. Dmitri and Viktor swap items.\n7. Viktor and Tariq swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Dmitri\": \"silver ring\", \"Viktor\": \"copper dice\", \"Yara\": \"ochre mirror\", \"Tariq\": \"indigo coin\"}, \"people\": [\"Dmitri\", \"Viktor\", \"Yara\", \"Tariq\"]}"
 },
 {
  "task_id": "capacity_hard_019",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Leif holds a jade bell\n- Joaquin holds a indigo pendant\n- Greta holds a cobalt feather\n- Gael holds a blue mirror\n\nSwaps:\n1. Joaquin and Greta swap items.\n2. Leif and Joaquin swap items.\n3. Leif and Gael swap items.\n4. Greta and Gael swap items.\n5. Joaquin and Greta swap items.\n6. Leif and Greta swap items.\n7. Joaquin and Gael swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Leif\": \"jade bell\", \"Joaquin\": \"indigo pendant\", \"Greta\": \"blue mirror\", \"Gael\": \"cobalt feather\"}, \"people\": [\"Leif\", \"Joaquin\", \"Greta\", \"Gael\"]}"
 },
 {
  "task_id": "capacity_hard_020",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Zora holds a slate stone\n- Dmitri holds a golden coin\n- Haruto holds a onyx scroll\n- Greta holds a coral chalice\n\nSwaps:\n1. Dmitri and Greta swap items.\n2. Haruto and Greta swap items.\n3. Zora and Haruto swap items.\n4. Dmitri and Haruto swap items.\n5. Zora and Greta swap items.\n6. Haruto and Greta swap items.\n7. Dmitri and Greta swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Zora\": \"onyx scroll\", \"Dmitri\": \"coral chalice\", \"Haruto\": \"golden coin\", \"Greta\": \"slate stone\"}, \"people\": [\"Zora\", \"Dmitri\", \"Haruto\", \"Greta\"]}"
 },
 {
  "task_id": "capacity_hard_021",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Wren holds a green flask\n- Zain holds a bronze coin\n- Kenji holds a ochre locket\n- Femi holds a pearl book\n\nSwaps:\n1. Kenji and Femi swap items.\n2. Wren and Kenji swap items.\n3. Zain and Kenji swap items.\n4. Zain and Femi swap items.\n5. Wren and Zain swap items.\n6. Zain and Kenji swap items.\n7. Kenji and Femi swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Wren\": \"ochre locket\", \"Zain\": \"bronze coin\", \"Kenji\": \"green flask\", \"Femi\": \"pearl book\"}, \"people\": [\"Wren\", \"Zain\", \"Kenji\", \"Femi\"]}"
 },
 {
  "task_id": "capacity_hard_022",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Uma holds a red feather\n- Viktor holds a bronze book\n- Lumi holds a crimson pendant\n- Bashir holds a slate flask\n\nSwaps:\n1. Viktor and Bashir swap items.\n2. Lumi and Bashir swap items.\n3. Viktor and Bashir swap items.\n4. Viktor and Lumi swap items.\n5. Uma and Viktor swap items.\n6. Viktor and Bashir swap items.\n7. Uma and Viktor swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Uma\": \"slate flask\", \"Viktor\": \"bronze book\", \"Lumi\": \"crimson pendant\", \"Bashir\": \"red feather\"}, \"people\": [\"Uma\", \"Viktor\", \"Lumi\", \"Bashir\"]}"
 },
 {
  "task_id": "capacity_hard_023",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Uma holds a onyx stone\n- Joelle holds a crimson pendant\n- Adaeze holds a coral key\n- Bashir holds a jade lantern\n\nSwaps:\n1. Joelle and Adaeze swap items.\n2. Adaeze and Bashir swap items.\n3. Uma and Joelle swap items.\n4. Joelle and Adaeze swap items.\n5. Uma and Joelle swap items.\n6. Uma and Bashir swap items.\n7. Joelle and Adaeze swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Uma\": \"crimson pendant\", \"Joelle\": \"onyx stone\", \"Adaeze\": \"coral key\", \"Bashir\": \"jade lantern\"}, \"people\": [\"Uma\", \"Joelle\", \"Adaeze\", \"Bashir\"]}"
 },
 {
  "task_id": "capacity_expert_024",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Hana holds a teal flask\n- Viktor holds a blue candle\n- Joaquin holds a russet mirror\n- Yuki holds a cobalt locket\n- Haruto holds a silver compass\n\nSwaps:\n1. Viktor and Yuki swap items.\n2. Joaquin and Haruto swap items.\n3. Yuki and Haruto swap items.\n4. Hana and Haruto swap items.\n5. Hana and Yuki swap items.\n6. Joaquin and Yuki swap items.\n7. Hana and Joaquin swap items.\n8. Hana and Viktor swap items.\n9. Hana and Joaquin swap items.\n10. Viktor and Joaquin swap items.\n11. Yuki and Haruto swap items.\n12. Hana and Joaquin swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Hana\": \"blue candle\", \"Viktor\": \"cobalt locket\", \"Joaquin\": \"russet mirror\", \"Yuki\": \"teal flask\", \"Haruto\": \"silver compass\"}, \"people\": [\"Hana\", \"Viktor\", \"Joaquin\", \"Yuki\", \"Haruto\"]}"
 },
 {
  "task_id": "capacity_expert_025",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Tariq holds a ochre chalice\n- Nalini holds a amber dice\n- Priya holds a crimson book\n- Elio holds a pearl locket\n- Xander holds a onyx pendant\n\nSwaps:\n1. Tariq and Nalini swap items.\n2. Tariq and Priya swap items.\n3. Tariq and Xander swap items.\n4. Tariq and Elio swap items.\n5. Elio and Xander swap items.\n6. Tariq and Xander swap items.\n7. Nalini and Elio swap items.\n8. Tariq and Elio swap items.\n9. Priya and Elio swap items.\n10. Nalini and Priya swap items.\n11. Priya and Xander swap items.\n12. Elio and Xander swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Tariq\": \"ochre chalice\", \"Nalini\": \"onyx pendant\", \"Priya\": \"pearl locket\", \"Elio\": \"crimson book\", \"Xander\": \"amber dice\"}, \"people\": [\"Tariq\", \"Nalini\", \"Priya\", \"Elio\", \"Xander\"]}"
 },
 {
  "task_id": "capacity_expert_026",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Bashir holds a red lantern\n- Hana holds a golden locket\n- Joaquin holds a blue feather\n- Uma holds a amber stone\n- Ravi holds a pearl coin\n\nSwaps:\n1. Hana and Ravi swap items.\n2. Joaquin and Uma swap items.\n3. Bashir and Hana swap items.\n4. Bashir and Joaquin swap items.\n5. Joaquin and Uma swap items.\n6. Hana and Ravi swap items.\n7. Bashir and Uma swap items.\n8. Bashir and Joaquin swap items.\n9. Hana and Joaquin swap items.\n10. Uma and Ravi swap items.\n11. Bashir and Joaquin swap items.\n12. Uma and Ravi swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Bashir\": \"golden locket\", \"Hana\": \"pearl coin\", \"Joaquin\": \"blue feather\", \"Uma\": \"amber stone\", \"Ravi\": \"red lantern\"}, \"people\": [\"Bashir\", \"Hana\", \"Joaquin\", \"Uma\", \"Ravi\"]}"
 },
 {
  "task_id": "capacity_expert_027",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Leif holds a pearl book\n- Kenji holds a silver bell\n- Bashir holds a ivory pendant\n- Viktor holds a red lantern\n- Zain holds a onyx ring\n\nSwaps:\n1. Kenji and Bashir swap items.\n2. Leif and Bashir swap items.\n3. Kenji and Bashir swap items.\n4. Leif and Bashir swap items.\n5. Leif and Kenji swap items.\n6. Kenji and Viktor swap items.\n7. Kenji and Zain swap items.\n8. Bashir and Viktor swap items.\n9. Bashir and Zain swap items.\n10. Bashir and Viktor swap items.\n11. Leif and Zain swap items.\n12. Kenji and Viktor swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Leif\": \"ivory pendant\", \"Kenji\": \"red lantern\", \"Bashir\": \"silver bell\", \"Viktor\": \"onyx ring\", \"Zain\": \"pearl book\"}, \"people\": [\"Leif\", \"Kenji\", \"Bashir\", \"Viktor\", \"Zain\"]}"
 },
 {
  "task_id": "capacity_expert_028",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Nalini holds a red bell\n- Tala holds a ivory lantern\n- Orla holds a silver pendant\n- Uma holds a ochre book\n- Soren holds a amber dice\n\nSwaps:\n1. Nalini and Soren swap items.\n2. Uma and Soren swap items.\n3. Nalini and Tala swap items.\n4. Nalini and Soren swap items.\n5. Tala and Orla swap items.\n6. Tala and Soren swap items.\n7. Orla and Uma swap items.\n8. Tala and Soren swap items.\n9. Nalini and Orla swap items.\n10. Nalini and Soren swap items.\n11. Uma and Soren swap items.\n12. Nalini and Soren swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Nalini\": \"amber dice\", \"Tala\": \"silver pendant\", \"Orla\": \"ochre book\", \"Uma\": \"red bell\", \"Soren\": \"ivory lantern\"}, \"people\": [\"Nalini\", \"Tala\", \"Orla\", \"Uma\", \"Soren\"]}"
 },
 {
  "task_id": "capacity_expert_029",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Gael holds a russet dagger\n- Viktor holds a coral locket\n- Lumi holds a teal bell\n- Elara holds a red key\n- Qadir holds a slate mask\n\nSwaps:\n1. Gael and Elara swap items.\n2. Gael and Qadir swap items.\n3. Gael and Lumi swap items.\n4. Gael and Qadir swap items.\n5. Lumi and Elara swap items.\n6. Gael and Viktor swap items.\n7. Elara and Qadir swap items.\n8. Viktor and Qadir swap items.\n9. Gael and Elara swap items.\n10. Lumi and Qadir swap items.\n11. Viktor and Qadir swap items.\n12. Gael and Viktor swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Gael\": \"russet dagger\", \"Viktor\": \"teal bell\", \"Lumi\": \"red key\", \"Elara\": \"coral locket\", \"Qadir\": \"slate mask\"}, \"people\": [\"Gael\", \"Viktor\", \"Lumi\", \"Elara\", \"Qadir\"]}"
 },
 {
  "task_id": "capacity_expert_030",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Amara holds a crimson chalice\n- Olena holds a indigo pendant\n- Orla holds a silver ring\n- Priya holds a slate coin\n- Colette holds a jade shell\n\nSwaps:\n1. Orla and Colette swap items.\n2. Priya and Colette swap items.\n3. Orla and Priya swap items.\n4. Amara and Colette swap items.\n5. Amara and Orla swap items.\n6. Amara and Olena swap items.\n7. Amara and Orla swap items.\n8. Olena and Colette swap items.\n9. Amara and Colette swap items.\n10. Olena and Orla swap items.\n11. Amara and Colette swap items.\n12. Olena and Colette swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Amara\": \"slate coin\", \"Olena\": \"silver ring\", \"Orla\": \"crimson chalice\", \"Priya\": \"jade shell\", \"Colette\": \"indigo pendant\"}, \"people\": [\"Amara\", \"Olena\", \"Orla\", \"Priya\", \"Colette\"]}"
 },
 {
  "task_id": "capacity_expert_031",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Hana holds a ivory mirror\n- Leif holds a jade dice\n- Soren holds a pearl pendant\n- Wren holds a onyx bell\n- Idris holds a blue ring\n\nSwaps:\n1. Soren and Wren swap items.\n2. Hana and Wren swap items.\n3. Hana and Idris swap items.\n4. Leif and Soren swap items.\n5. Wren and Idris swap items.\n6. Soren and Idris swap items.\n7. Hana and Soren swap items.\n8. Leif and Idris swap items.\n9. Hana and Wren swap items.\n10. Soren and Idris swap items.\n11. Hana and Soren swap items.\n12. Wren and Idris swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Hana\": \"onyx bell\", \"Leif\": \"jade dice\", \"Soren\": \"pearl pendant\", \"Wren\": \"blue ring\", \"Idris\": \"ivory mirror\"}, \"people\": [\"Hana\", \"Leif\", \"Soren\", \"Wren\", \"Idris\"]}"
 },
 {
  "task_id": "capacity_frontier_032",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Nalini holds a onyx locket\n- Greta holds a slate pendant\n- Orla holds a indigo stone\n- Paloma holds a cobalt key\n- Celine holds a teal feather\n- Leif holds a bronze chalice\n- Qadir holds a golden scroll\n- Kaia holds a ochre flask\n\nSwaps:\n1. Celine and Leif swap items.\n2. Greta and Orla swap items.\n3. Paloma and Kaia swap items.\n4. Qadir and Kaia swap items.\n5. Nalini and Celine swap items.\n6. Celine and Leif swap items.\n7. Orla and Qadir swap items.\n8. Celine and Leif swap items.\n9. Nalini and Orla swap items.\n10. Greta and Celine swap items.\n11. Paloma and Leif swap items.\n12. Orla and Celine swap items.\n13. Leif and Kaia swap items.\n14. Nalini and Qadir swap items.\n15. Greta and Leif swap items.\n16. Leif and Kaia swap items.\n17. Nalini and Celine swap items.\n18. Greta and Paloma swap items.\n19. Nalini and Qadir swap items.\n20. Nalini and Celine swap items.\n21. Orla and Leif swap items.\n22. Nalini and Leif swap items.\n23. Nalini and Orla swap items.\n24. Paloma and Leif swap items.\n25. Orla and Kaia swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Nalini\": \"ochre flask\", \"Greta\": \"teal feather\", \"Orla\": \"onyx locket\", \"Paloma\": \"slate pendant\", \"Celine\": \"cobalt key\", \"Leif\": \"golden scroll\", \"Qadir\": \"bronze chalice\", \"Kaia\": \"indigo stone\"}, \"people\": [\"Nalini\", \"Greta\", \"Orla\", \"Paloma\", \"Celine\", \"Leif\", \"Qadir\", \"Kaia\"]}"
 },
 {
  "task_id": "capacity_frontier_033",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Olena holds a golden flask\n- Runa holds a jade chalice\n- Nico holds a bronze shell\n- Lumi holds a amber key\n- Joelle holds a crimson scroll\n- Sigrid holds a indigo bell\n- Bram holds a onyx dice\n- Dmitri holds a pearl book\n\nSwaps:\n1. Olena and Runa swap items.\n2. Runa and Dmitri swap items.\n3. Runa and Bram swap items.\n4. Bram and Dmitri swap items.\n5. Sigrid and Dmitri swap items.\n6. Nico and Lumi swap items.\n7. Lumi and Dmitri swap items.\n8. Nico and Dmitri swap items.\n9. Runa and Nico swap items.\n10. Bram and Dmitri swap items.\n11. Olena and Runa swap items.\n12. Joelle and Sigrid swap items.\n13. Lumi and Joelle swap items.\n14. Lumi and Bram swap items.\n15. Nico and Bram swap items.\n16. Nico and Dmitri swap items.\n17. Lumi and Bram swap items.\n18. Sigrid and Dmitri swap items.\n19. Runa and Bram swap items.\n20. Olena and Runa swap items.\n21. Bram and Dmitri swap items.\n22. Nico and Sigrid swap items.\n23. Olena and Nico swap items.\n24. Runa and Lumi swap items.\n25. Joelle and Bram swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Olena\": \"pearl book\", \"Runa\": \"onyx dice\", \"Nico\": \"amber key\", \"Lumi\": \"bronze shell\", \"Joelle\": \"crimson scroll\", \"Sigrid\": \"golden flask\", \"Bram\": \"indigo bell\", \"Dmitri\": \"jade chalice\"}, \"people\": [\"Olena\", \"Runa\", \"Nico\", \"Lumi\", \"Joelle\", \"Sigrid\", \"Bram\", \"Dmitri\"]}"
 },
 {
  "task_id": "capacity_frontier_034",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Dmitri holds a jade scroll\n- Viktor holds a russet mirror\n- Hana holds a silver pendant\n- Yuki holds a golden stone\n- Gael holds a cobalt feather\n- Wren holds a pearl lantern\n- Paloma holds a bronze chalice\n- Yara holds a blue candle\n\nSwaps:\n1. Dmitri and Paloma swap items.\n2. Hana and Gael swap items.\n3. Dmitri and Gael swap items.\n4. Yuki and Yara swap items.\n5. Hana and Wren swap items.\n6. Dmitri and Hana swap items.\n7. Viktor and Yuki swap items.\n8. Viktor and Wren swap items.\n9. Viktor and Gael swap items.\n10. Dmitri and Wren swap items.\n11. Wren and Yara swap items.\n12. Viktor and Yuki swap items.\n13. Gael and Yara swap items.\n14. Yuki and Gael swap items.\n15. Dmitri and Wren swap items.\n16. Hana and Wren swap items.\n17. Dmitri and Yuki swap items.\n18. Viktor and Yara swap items.\n19. Hana and Yuki swap items.\n20. Viktor and Gael swap items.\n21. Hana and Gael swap items.\n22. Yuki and Wren swap items.\n23. Viktor and Paloma swap items.\n24. Dmitri and Wren swap items.\n25. Viktor and Yara swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Dmitri\": \"blue candle\", \"Viktor\": \"russet mirror\", \"Hana\": \"cobalt feather\", \"Yuki\": \"silver pendant\", \"Gael\": \"golden stone\", \"Wren\": \"pearl lantern\", \"Paloma\": \"bronze chalice\", \"Yara\": \"jade scroll\"}, \"people\": [\"Dmitri\", \"Viktor\", \"Hana\", \"Yuki\", \"Gael\", \"Wren\", \"Paloma\", \"Yara\"]}"
 },
 {
  "task_id": "capacity_frontier_035",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Soren holds a amber mirror\n- Ines holds a russet lantern\n- Ugo holds a onyx scroll\n- Zora holds a slate locket\n- Magnus holds a ochre mask\n- Colette holds a indigo feather\n- Hana holds a bronze ring\n- Uma holds a ivory shell\n\nSwaps:\n1. Hana and Uma swap items.\n2. Ugo and Zora swap items.\n3. Soren and Hana swap items.\n4. Colette and Uma swap items.\n5. Magnus and Hana swap items.\n6. Soren and Magnus swap items.\n7. Soren and Zora swap items.\n8. Ugo and Zora swap items.\n9. Ugo and Uma swap items.\n10. Soren and Ugo swap items.\n11. Hana and Uma swap items.\n12. Zora and Colette swap items.\n13. Soren and Ugo swap items.\n14. Colette and Hana swap items.\n15. Zora and Hana swap items.\n16. Zora and Magnus swap items.\n17. Soren and Magnus swap items.\n18. Colette and Hana swap items.\n19. Ugo and Colette swap items.\n20. Magnus and Colette swap items.\n21. Ugo and Zora swap items.\n22. Magnus and Colette swap items.\n23. Zora and Colette swap items.\n24. Hana and Uma swap items.\n25. Magnus and Hana swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Soren\": \"slate locket\", \"Ines\": \"russet lantern\", \"Ugo\": \"ivory shell\", \"Zora\": \"indigo feather\", \"Magnus\": \"ochre mask\", \"Colette\": \"bronze ring\", \"Hana\": \"onyx scroll\", \"Uma\": \"amber mirror\"}, \"people\": [\"Soren\", \"Ines\", \"Ugo\", \"Zora\", \"Magnus\", \"Colette\", \"Hana\", \"Uma\"]}"
 },
 {
  "task_id": "capacity_frontier_036",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Joaquin holds a jade chalice\n- Elara holds a ivory candle\n- Ugo holds a ochre pendant\n- Kenji holds a blue flask\n- Zain holds a coral coin\n- Adaeze holds a amber key\n- Tariq holds a silver book\n- Lumi holds a indigo feather\n\nSwaps:\n1. Elara and Tariq swap items.\n2. Elara and Lumi swap items.\n3. Adaeze and Lumi swap items.\n4. Kenji and Zain swap items.\n5. Joaquin and Kenji swap items.\n6. Elara and Tariq swap items.\n7. Zain and Adaeze swap items.\n8. Elara and Zain swap items.\n9. Ugo and Lumi swap items.\n10. Zain and Adaeze swap items.\n11. Joaquin and Adaeze swap items.\n12. Joaquin and Lumi swap items.\n13. Adaeze and Tariq swap items.\n14. Ugo and Zain swap items.\n15. Joaquin and Adaeze swap items.\n16. Kenji and Lumi swap items.\n17. Adaeze and Lumi swap items.\n18. Joaquin and Tariq swap items.\n19. Kenji and Tariq swap items.\n20. Kenji and Adaeze swap items.\n21. Joaquin and Elara swap items.\n22. Elara and Kenji swap items.\n23. Kenji and Tariq swap items.\n24. Elara and Ugo swap items.\n25. Joaquin and Tariq swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Joaquin\": \"coral coin\", \"Elara\": \"blue flask\", \"Ugo\": \"jade chalice\", \"Kenji\": \"ivory candle\", \"Zain\": \"amber key\", \"Adaeze\": \"indigo feather\", \"Tariq\": \"silver book\", \"Lumi\": \"ochre pendant\"}, \"people\": [\"Joaquin\", \"Elara\", \"Ugo\", \"Kenji\", \"Zain\", \"Adaeze\", \"Tariq\", \"Lumi\"]}"
 },
 {
  "task_id": "capacity_frontier_037",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Ravi holds a russet mask\n- Bram holds a blue chalice\n- Kenji holds a pearl dagger\n- Kaia holds a golden feather\n- Zain holds a cobalt shell\n- Haruto holds a slate book\n- Willa holds a green locket\n- Dariush holds a jade key\n\nSwaps:\n1. Kenji and Kaia swap items.\n2. Bram and Zain swap items.\n3. Bram and Haruto swap items.\n4. Bram and Zain swap items.\n5. Zain and Haruto swap items.\n6. Ravi and Bram swap items.\n7. Zain and Willa swap items.\n8. Bram and Willa swap items.\n9. Ravi and Dariush swap items.\n10. Haruto and Willa swap items.\n11. Bram and Haruto swap items.\n12. Kenji and Haruto swap items.\n13. Haruto and Dariush swap items.\n14. Ravi and Kenji swap items.\n15. Bram and Zain swap items.\n16. Haruto and Dariush swap items.\n17. Kaia and Zain swap items.\n18. Ravi and Kenji swap items.\n19. Zain and Dariush swap items.\n20. Kenji and Kaia swap items.\n21. Kenji and Willa swap items.\n22. Zain and Willa swap items.\n23. Ravi and Haruto swap items.\n24. Zain and Haruto swap items.\n25. Kaia and Willa swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Ravi\": \"golden feather\", \"Bram\": \"green locket\", \"Kenji\": \"slate book\", \"Kaia\": \"blue chalice\", \"Zain\": \"jade key\", \"Haruto\": \"russet mask\", \"Willa\": \"cobalt shell\", \"Dariush\": \"pearl dagger\"}, \"people\": [\"Ravi\", \"Bram\", \"Kenji\", \"Kaia\", \"Zain\", \"Haruto\", \"Willa\", \"Dariush\"]}"
 },
 {
  "task_id": "capacity_frontier_038",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Joaquin holds a bronze book\n- Ines holds a teal chalice\n- Nalini holds a coral locket\n- Adaeze holds a onyx ring\n- Sigrid holds a slate feather\n- Idris holds a blue lantern\n- Greta holds a ochre coin\n- Bashir holds a pearl pendant\n\nSwaps:\n1. Joaquin and Bashir swap items.\n2. Joaquin and Ines swap items.\n3. Ines and Bashir swap items.\n4. Sigrid and Bashir swap items.\n5. Ines and Greta swap items.\n6. Adaeze and Sigrid swap items.\n7. Sigrid and Bashir swap items.\n8. Idris and Greta swap items.\n9. Ines and Sigrid swap items.\n10. Joaquin and Adaeze swap items.\n11. Nalini and Idris swap items.\n12. Joaquin and Ines swap items.\n13. Ines and Adaeze swap items.\n14. Adaeze and Greta swap items.\n15. Idris and Bashir swap items.\n16. Greta and Bashir swap items.\n17. Nalini and Adaeze swap items.\n18. Ines and Adaeze swap items.\n19. Joaquin and Ines swap items.\n20. Ines and Bashir swap items.\n21. Joaquin and Ines swap items.\n22. Joaquin and Greta swap items.\n23. Ines and Bashir swap items.\n24. Ines and Sigrid swap items.\n25. Ines and Bashir swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Joaquin\": \"coral locket\", \"Ines\": \"bronze book\", \"Nalini\": \"blue lantern\", \"Adaeze\": \"teal chalice\", \"Sigrid\": \"slate feather\", \"Idris\": \"onyx ring\", \"Greta\": \"pearl pendant\", \"Bashir\": \"ochre coin\"}, \"people\": [\"Joaquin\", \"Ines\", \"Nalini\", \"Adaeze\", \"Sigrid\", \"Idris\", \"Greta\", \"Bashir\"]}"
 },
 {
  "task_id": "capacity_frontier_039",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Tariq holds a slate chalice\n- Qadir holds a onyx feather\n- Yuki holds a blue scroll\n- Tala holds a pearl mask\n- Zora holds a cobalt locket\n- Ravi holds a jade coin\n- Gael holds a indigo compass\n- Colette holds a silver flask\n\nSwaps:\n1. Tala and Ravi swap items.\n2. Tala and Zora swap items.\n3. Yuki and Colette swap items.\n4. Tala and Zora swap items.\n5. Tariq and Gael swap items.\n6. Tala and Ravi swap items.\n7. Tariq and Zora swap items.\n8. Qadir and Tala swap items.\n9. Tariq and Colette swap items.\n10. Tariq and Qadir swap items.\n11. Qadir and Yuki swap items.\n12. Tariq and Colette swap items.\n13. Tala and Ravi swap items.\n14. Tariq and Tala swap items.\n15. Gael and Colette swap items.\n16. Tariq and Zora swap items.\n17. Ravi and Colette swap items.\n18. Tariq and Gael swap items.\n19. Zora and Ravi swap items.\n20. Tala and Colette swap items.\n21. Tala and Ravi swap items.\n22. Ravi and Colette swap items.\n23. Yuki and Ravi swap items.\n24. Tariq and Colette swap items.\n25. Tariq and Gael swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Tariq\": \"indigo compass\", \"Qadir\": \"silver flask\", \"Yuki\": \"cobalt locket\", \"Tala\": \"jade coin\", \"Zora\": \"slate chalice\", \"Ravi\": \"blue scroll\", \"Gael\": \"onyx feather\", \"Colette\": \"pearl mask\"}, \"people\": [\"Tariq\", \"Qadir\", \"Yuki\", \"Tala\", \"Zora\", \"Ravi\", \"Gael\", \"Colette\"]}"
 }
]
''')

print(f"Loaded {len(DATASET)} items")
for tt in ['capacity']:
    count = sum(1 for d in DATASET if d["task_type"] == tt)
    print(f"  {tt}: {count} items")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Execution Loop
# ══════════════════════════════════════════════════════════════════════

TASK_DISPATCH = {
    "capacity": cogattention_capacity,
}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Attention Capacity")
